In [11]:
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone, timedelta
import json
import sqlite3
import hashlib


@dataclass(eq=True, frozen=True)
class VesselMetadataKey:
    unique_vessel_id: str
    obs_time: datetime

def sort_dict_list_by_keys(input: list, output: list) -> list:
    for e in input:
        if isinstance(e, dict):
            d = {}
            output.append(d)
            sort_dict_by_keys(e, d)
        elif isinstance(e, list):
            l = []
            output.append(l)
            sort_dict_list_by_keys(e, l)
        else:
            output.append(e)

def sort_dict_by_keys(input: dict, output: dict, *,
                      exclude_keys: tuple = ('time', 'fileType', 'submissionInfo', 'dataProcessed')) -> dict:
    for k, v in sorted(input.items()):
        if k in exclude_keys:
            continue
        if isinstance(v, dict):
            output[k] = {}
            sort_dict_by_keys(v, output[k])
        elif isinstance(v, list):
            output[k] = []
            sort_dict_list_by_keys(v, output[k])
        else:
            output[k] = v
    return output

def get_unique_vessel_id(data: dict) -> str|None:
    if 'platform' in data:
        platform = data['platform']
        if 'uniqueID' in platform:
            uniqueId = platform['uniqueID']
            if not isinstance(uniqueId, str):
                raise ValueError(f"Expected uniqueID to be of type str but is of type {type(uniqueId)}")
            return uniqueId
    if 'trustedNode' in data:
        trustedNode = data['trustedNode']
        if 'uniqueVesselID' in trustedNode:
            uniqueVesselId = trustedNode['uniqueVesselID']
            if not isinstance(uniqueVesselId, str):
                raise ValueError(f"Expected uniqueVesselId to be of type str but is of type {type(uniqueVesselId)}")
            return uniqueVesselId
    return None

def parse_timestamp(time_in: int) -> datetime:
    if not isinstance(time_in, int):
        raise ValueError(f"Expected input time {time_in} to be of type int, but it was of type {type(time_in)}")
    seconds, ms = divmod(time_in, 1000)
    dt = datetime.fromtimestamp(seconds, tz=timezone.utc)
    delta_ms = timedelta(milliseconds=ms)
    return dt + delta_ms

def get_start_end_times(data: dict) -> tuple[datetime, datetime]|tuple[None, None]:
    if 'time' in data:
        time_block = data['time']
        if 'startTime' not in time_block or 'endTime' not in time_block:
            return None, None
        try:
            start_time = parse_timestamp(time_block['startTime'])
        except ValueError:
            return None, None
        try:
            end_time = parse_timestamp(time_block['endTime'])
        except ValueError:
            return None, None
        return start_time, end_time
    return None, None

In [43]:
# doc_root should be local path to https://github.com/CCOMJHC/csbschema/tree/main/docs/IHO
# doc_root: Path = Path('../docs/IHO')
# example_docs: list[str] = [
#     'b12_v3_1_0_example-2023-08.json'
# ]


In [2]:
doc_root: Path = Path('example1')
for doc in doc_root.glob('*.json'):
    print(str(doc))

example1/A15260F9-CE28-41FB-AED8-B91AB52D1B78.json
example1/F655ADEC-E1D8-4F79-BF98-0C05D58FFF52.json
example1/0A99C49B-8262-40FC-BFB6-2F21848B0299.json
example1/698F8CE4-C8D5-4FD2-8AEC-F64767258CB9.json
example1/5BAB57D1-7D21-4E77-8497-2524E718D012.json


In [17]:
vessel_meta: dict[VesselMetadataKey, dict] = {}

for doc in doc_root.glob('*.json'):
    # print(str(doc))
    with doc.open(mode='rt') as f:
        doc_data: dict = json.load(f)

    try:
        uniqueId = get_unique_vessel_id(doc_data)
    except ValueError as e:
        print(f"Unable to read unique ID for file {str(doc)} due to error {str(e)}, skipping...")
        continue
    if uniqueId is None:
        print(f"No unique ID for file {str(doc)}, skipping...")
        continue

    print(f"Unique Id for file {str(doc)} is {uniqueId}")

    try:
        start_time, end_time = get_start_end_times(doc_data)
    except ValueError as e:
        print(f"Unable to read start,end time for file {str(doc)} due to error {str(e)}, skipping...")
        continue
    if start_time is None or end_time is None:
        print(f"Expected start and end time for file {str(doc)} to not be None, but one of them was None, skipping...")
        continue

    print(f"start, end time for file {str(doc)} is {start_time}, {end_time}.")

    # print(f"raw doc_meta: {json.dumps(doc_meta)}\n\n")
    doc_meta: dict = sort_dict_by_keys(doc_data, {})
    # print(f"sorted doc_meta: {json.dumps(doc_meta)}\n\n")
    key = VesselMetadataKey(
        unique_vessel_id=uniqueId,
        obs_time=start_time
    )
    vessel_meta[key] = doc_meta

print(vessel_meta)

Unique Id for file example1/A15260F9-CE28-41FB-AED8-B91AB52D1B78.json is OFM-72b748d0-9890-11f0-bdd5-b1670927a4f1
start, end time for file example1/A15260F9-CE28-41FB-AED8-B91AB52D1B78.json is 2026-02-04 20:03:23+00:00, 2026-02-04 21:03:34+00:00.
Unique Id for file example1/F655ADEC-E1D8-4F79-BF98-0C05D58FFF52.json is SIGNALK-ac020bdf-5c0e-4c82-844f-1db2bc73383a
start, end time for file example1/F655ADEC-E1D8-4F79-BF98-0C05D58FFF52.json is 2022-01-03 12:06:31+00:00, 2022-01-03 17:59:59+00:00.
Unique Id for file example1/0A99C49B-8262-40FC-BFB6-2F21848B0299.json is PGS-834f85c2-0999-11eb-a100-98be942a5b5a
start, end time for file example1/0A99C49B-8262-40FC-BFB6-2F21848B0299.json is 2026-02-02 19:00:00+00:00, 2026-02-02 19:00:00+00:00.
Unique Id for file example1/698F8CE4-C8D5-4FD2-8AEC-F64767258CB9.json is AQM-687ce9f49cea48-68471861
start, end time for file example1/698F8CE4-C8D5-4FD2-8AEC-F64767258CB9.json is 2026-01-22 20:12:18+00:00, 2026-01-22 20:30:47+00:00.
Unique Id for file ex

In [18]:
con = sqlite3.connect('vessel_meta.db')
cur = con.cursor()
cur.execute('CREATE TABLE vessels(unique_vessel_id TEXT, obs_time DATETIME, hash TEXT, metadata JSON, PRIMARY KEY(unique_vessel_id, obs_time))')

In [19]:
for k, v in vessel_meta.items():
    m = hashlib.sha3_256()
    metadata: str = json.dumps(v)
    m.update(bytes(metadata, 'utf-8'))
    metadata_hash: str = m.hexdigest()
    data = [k.unique_vessel_id, k.obs_time, metadata_hash, metadata]
    cur.execute("INSERT INTO vessels VALUES(?, ?, ?, ?)", data)

/var/folders/9k/23swhcmn1_jg4y1hpnkkt5l00000gs/T/ipykernel_60129/3211672124.py:7: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cur.execute("INSERT INTO vessels VALUES(?, ?, ?, ?)", data)


In [20]:
res = cur.execute('SELECT * FROM vessels')
res.fetchall()

[('OFM-72b748d0-9890-11f0-bdd5-b1670927a4f1',
  '2026-02-04 20:03:23+00:00',
  'bbf8c7d8c5aaadb6a16077547c1a4d2bf768987119b1927d57c202e7976ad6cb',
  '{"convention": "XYZ CSB 3.0", "correctors": {"draftApplied": "False", "motionOffsetsApplied": "False", "positionOffsetDocumented": "False", "positionReferencePoint": "GNSS", "soundSpeedDocumented": "False"}, "crs": {"horizontal": {"type": "EPSG", "value": 4326}, "vertical": "Transducer"}, "dataLicense": "CCO 1.0", "platform": {"IDNumber": "MNZ132394", "IDType": "MMSI", "length": 7.8, "name": "NIWA Orca", "sensors": [{"draft": 0.42, "make": "Simrad", "model": "S3D-65.1.14", "position": "[-2.6200,0.7100,2.5920]", "transducer": "n/a", "type": "Sounder"}, {"make": "Simrad", "model": "GS25 Antenna (102472#)", "position": "[0,0,0]", "type": "GNSS"}, {"make": "CSS Electronics", "model": "CANmod.gps", "position": "[1.9950,0.2150,0.0000]", "type": "MotionSensor"}], "type": "Government", "uniqueID": "OFM-72b748d0-9890-11f0-bdd5-b1670927a4f1"}, "pro